# Steps 4-5: NLP Preprocessing & Feature Extraction

Text cleaning, sentiment analysis (VADER/TextBlob), TF-IDF, and occupation clustering.

In [ ]:
import sys
sys.path.insert(0, '/home/jovyan/work/src')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from spark_utils import get_spark_session, save_parquet
from nlp_pipeline import clean_text, add_sentiment_features, build_tfidf_features, cluster_occupations, create_missing_text_flag

In [ ]:
SAMPLE_MODE = True

## 4.1 Load Baseline Features

In [ ]:
spark = get_spark_session()
df = spark.read.parquet('/home/jovyan/work/data/parquet/features_baseline.parquet')
df_pd = df.toPandas()

## 4.2 Text Cleaning Demo

In [ ]:
# Before/after examples
sample_texts = df_pd['desc'].dropna().head(5)
for text in sample_texts:
    print('BEFORE:', text[:100])
    print('AFTER :', clean_text(text)[:100])
    print('---')

## 4.3 Missing Text Flags

In [ ]:
df_pd = create_missing_text_flag(df_pd, 'desc')
df_pd = create_missing_text_flag(df_pd, 'title')
print(f"desc missing rate: {df_pd['desc_missing'].mean():.2%}")
print(f"title missing rate: {df_pd['title_missing'].mean():.2%}")

## 4.4 Sentiment Analysis

In [ ]:
df_pd = add_sentiment_features(df_pd, 'desc')
df_pd = add_sentiment_features(df_pd, 'title')

In [ ]:
# Sentiment distribution plots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df_pd['desc_vader'].dropna(), bins=50, edgecolor='black')
axes[0].set_title('VADER Sentiment: desc')
axes[0].set_xlabel('Compound Score')

axes[1].hist(df_pd['title_vader'].dropna(), bins=50, edgecolor='black')
axes[1].set_title('VADER Sentiment: title')
axes[1].set_xlabel('Compound Score')

plt.tight_layout()
plt.savefig('/home/jovyan/work/outputs/figures/sentiment_distribution.png', dpi=150)
plt.show()

## 4.5 TF-IDF Features

In [ ]:
# TF-IDF on desc (non-missing only, fill missing with empty string)
cleaned_desc = df_pd['desc'].apply(clean_text)
tfidf_matrix, feature_names, vectorizer = build_tfidf_features(cleaned_desc, max_features=50)

# Add TF-IDF columns to dataframe
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=[f'tfidf_{name}' for name in feature_names])
df_pd = pd.concat([df_pd.reset_index(drop=True), tfidf_df], axis=1)

print(f'Added {len(feature_names)} TF-IDF features')
print('Top terms:', list(feature_names[:10]))

## 4.6 Occupation Clustering

In [ ]:
df_pd['occupation_cluster'] = cluster_occupations(df_pd['emp_title'], n_clusters=20)
print(f'Unique clusters: {df_pd["occupation_cluster"].nunique()}')
df_pd['occupation_cluster'].value_counts().head(10)

## 4.7 Save Hybrid Feature Matrix

In [ ]:
hybrid_df = spark.createDataFrame(df_pd)
save_parquet(hybrid_df, '/home/jovyan/work/data/parquet/features_hybrid.parquet')
print('Hybrid features saved.')